### Testing Logits

In [1]:
# Logprob is just log(softmax(logits))
# But only top-k logprobs is given by cloud llm models

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import math

load_dotenv()
client = OpenAI(
    api_key=os.environ.get("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)

response = client.chat.completions.create(
    model="deepseek-v4-flash",  
    messages=[{"role": "user", "content": "Is the sky blue? Answer yes or no."}],
    logprobs=True,
    top_logprobs=5   #max 20
)

# Per-token logprobs
for token_info in response.choices[0].logprobs.content:
    prob = math.exp(token_info.logprob)
    print(f"Token: {token_info.token!r:15} logprob: {token_info.logprob:.3f}  prob: {prob:.3f}")
    
    # Top alternatives at this position
    for alt in token_info.top_logprobs:
        print(f"  alt: {alt.token!r:15} logprob: {alt.logprob:.3f}")

Token: 'yes'           logprob: -0.334  prob: 0.716
  alt: 'yes'           logprob: -0.334
  alt: 'Yes'           logprob: -1.259
  alt: '<｜｜end▁of▁sentence｜｜>' logprob: -14.642
  alt: 'y'             logprob: -14.994
  alt: ' yes'          logprob: -15.345


### Getting chain-of-thought chunks

In [ ]:
messages = [{"role": "user", "content": "9.11 and 9.8, which is greater?"}]

response = client.chat.completions.create(
    model="deepseek-v4-pro",
    messages=messages,
    reasoning_effort="high",
    extra_body={"thinking": {"type": "enabled"}}
)

reasoning_content = response.choices[0].message.reasoning_content
content = response.choices[0].message.content

print(reasoning_content)
print(f"Final answer:\n {content}")



We need to compare 9.11 and 9.8. The user asks "9.11 and 9.8, which is greater?" This is a simple number comparison. 9.11 is nine and eleven hundredths, 9.8 is nine and eight tenths, which is nine and eighty hundredths. Since 80 > 11, 9.8 > 9.11. But sometimes people confuse decimal numbers with version numbers. For example, they might think 9.11 is greater than 9.8 because 11 > 8 if they think of them as separate integers (like version numbers). However, mathematically, 9.8 is greater. I'll answer clearly: 9.8 is greater than 9.11. Explain why if needed.
9.8 is greater than 9.11. 

This is a common point of confusion because people sometimes treat the numbers as version numbers (where 9.11 would come after 9.8), but mathematically, they are decimals:

- 9.8 = 9.80  
- 9.11 = 9.11  

Since 80 hundredths is greater than 11 hundredths, **9.8 > 9.11**.


### Monte Carlo sim